In [11]:
# CELL 1: Setup
from google.colab import drive
import pandas as pd
import torch
from transformers import pipeline
import gc

# Mount Google Drive
drive.mount('/content/drive')

# Check for GPU
device = 0 if torch.cuda.is_available() else -1
if device == 0:
    print("GPU is active! Inference will run quickly.")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type and select T4 GPU.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU is active! Inference will run quickly.


In [12]:
# CELL 2: Format Data
# UPDATE THIS PATH to your new unlabeled CSV file
csv_path = "/content/drive/MyDrive/Colab Notebooks/ncte_utterances_sampled_for_annotation.csv"
df = pd.read_csv(csv_path)

# Function to perfectly recreate Architecture 1
def format_arch1(row):
    # Safely handle any empty cells
    prev = str(row['previous_context']) if pd.notna(row['previous_context']) else ""
    utt = str(row['student_utterance']) if pd.notna(row['student_utterance']) else ""
    turn = str(row['turn']) if pd.notna(row['turn']) else ""
    subseq = str(row['subsequent_context']) if pd.notna(row['subsequent_context']) else ""

    # Reconstruct the exact string format the model expects
    return f"{prev} </s></s> [TARGET_START] ({turn}) [S] {utt} [TARGET_END] {subseq}"

print("Applying Architecture 1 formatting...")
df['formatted_text'] = df.apply(format_arch1, axis=1)

# Convert to a list for the Hugging Face pipeline
texts_to_score = df['formatted_text'].tolist()

print(f"Loaded {len(texts_to_score)} rows.")
print("\nExample input for the model:")
print(texts_to_score[0])

Applying Architecture 1 formatting...
Loaded 200 rows.

Example input for the model:
(114) [S] Oh.
(115) [T] Take your fraction strips out and test it.
(116) [S] Yes.
(117) [S] Found it.
(118) [T] One-half is less than – and please don’t tell me one whole, ’cause I want something different than one whole.
(119) [S] I had [inaudible].
(120) [T] Finish putting together quickly.  Student A, what do you think? One-half is – </s></s> [TARGET_START] (121) [S] [Inaudible] – [TARGET_END] (122) [T] Less than –
(123) [S] Less than [inaudible].
(124) [T] Are you – you’re using your strips.  Take out one-half, like I did, and find a strip that’s bigger.  It might be two strips.  It might be three strips.  What strips are bigger than one-half?
(125) [S] One whole.
(126) [S] No.
(127) [T] I said don’t use one whole.  That is true, but that’s a little too easy.  We want to be more challenged.
(128) [S] One [inaudible] one-fourth.


In [17]:
# CELL 3: Offering Math Help Inference
# UPDATE PATH to your final_model folder
math_help_path = "/content/drive/MyDrive/roberta_experiments/Offering_Math_Help_Lower_LR_Higher_Batch_Size/final_model"

print("Loading Math Help model to GPU...")
pipe_math = pipeline("text-classification", model=math_help_path, tokenizer=math_help_path, device=device)

print("Generating labels in batches...")
math_results = pipe_math(texts_to_score, batch_size=16, truncation=True, max_length=512)

# Extract predictions (Assuming LABEL_1 is your positive class)
df['Predicted_Math_Help'] = [1 if res['label'] == 'LABEL_1' else 0 for res in math_results]
print("Math Help inference complete!")

# VERY IMPORTANT: Free up GPU memory so the next model doesn't crash Colab
del pipe_math
gc.collect()
torch.cuda.empty_cache()

Loading Math Help model to GPU...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Generating labels in batches...
Math Help inference complete!


In [18]:
# CELL 4: Successful Uptake Inference
# UPDATE PATH to your final_model folder
uptake_path = "/content/drive/MyDrive/roberta_experiments/Successful_Uptake_Higher_LR/final_model"

print("Loading Successful Uptake model to GPU...")
pipe_uptake = pipeline("text-classification", model=uptake_path, tokenizer=uptake_path, device=device)

print("Generating labels in batches...")
uptake_results = pipe_uptake(texts_to_score, batch_size=16, truncation=True, max_length=512)

# Extract predictions
df['Predicted_Successful_Uptake'] = [1 if res['label'] == 'LABEL_1' else 0 for res in uptake_results]
print("Successful Uptake inference complete!")

# Free up memory
del pipe_uptake
gc.collect()
torch.cuda.empty_cache()

Loading Successful Uptake model to GPU...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Generating labels in batches...
Successful Uptake inference complete!


In [19]:
# CELL 5: Save and Review
# UPDATE THIS PATH to where you want the final CSV to be saved
output_filename = "/content/drive/MyDrive/Colab Notebooks/labeled_ncte_dataset.csv"

# Drop the 'formatted_text' column so your final CSV is clean and readable
df = df.drop(columns=['formatted_text'])
df.to_csv(output_filename, index=False)
print(f"Success! Final dataset saved to {output_filename}\n")

# Spot Check
print("--- SPOT CHECK: OFFERING MATH HELP (Showing up to 3) ---")
math_hits = df[df['Predicted_Math_Help'] == 1]
for idx, row in math_hits.head(3).iterrows():
    print(f"Turn {row['turn']}: {row['student_utterance']}")

print("\n--- SPOT CHECK: SUCCESSFUL UPTAKE (Showing up to 3) ---")
uptake_hits = df[df['Predicted_Successful_Uptake'] == 1]
for idx, row in uptake_hits.head(3).iterrows():
    print(f"Turn {row['turn']}: {row['student_utterance']}")

KeyError: "['formatted_text'] not found in axis"